# Space Race Analysis (1957 onwards)

This notebook loads and explores space mission data from 1957 onwards, cleans key fields, and creates visualizations to answer questions about:
- missions per year and by country
- cost trends over time
- most common months for launches
- success and failure rates
- dominant organizations.

**Instructions:** place the `space_missions.csv` data file in this folder or update the `DATA_PATH` variable. If you don't have the CSV, download the dataset provided for the exercise and put it here.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use('seaborn')
sns.set_theme(style='whitegrid')

PROJECT_DIR = Path(__file__).parent if '__file__' in globals() else Path('.')
DEFAULT_CSV_NAMES = ['space_missions.csv', 'space_race_launches.csv', 'launches.csv', 'space_data.csv']
DATA_PATH = None

def find_dataset():
    candidates = [PROJECT_DIR / name for name in DEFAULT_CSV_NAMES if (PROJECT_DIR / name).exists()]
    candidates += sorted(PROJECT_DIR.glob('*.csv'))
    candidates += sorted(PROJECT_DIR.glob('*.xlsx'))
    if not candidates:
        raise FileNotFoundError(
            'No data file found. Place the CSV in this folder or update DATA_PATH.'
        )
    return Path(candidates[0])

def load_space_data(data_path: Path | str | None = None) -> pd.DataFrame:
    path = Path(data_path) if data_path else find_dataset()
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path, low_memory=False)
    elif path.suffix.lower() in ['.xls', '.xlsx']:
        df = pd.read_excel(path)
    else:
        raise ValueError(f'Unsupported file format: {path.suffix}')
    return df

def describe_dataset(df: pd.DataFrame) -> None:
    print('Data path:', DATA_PATH or find_dataset())
    print('Dataset dimensions:', df.shape)
    display(df.head(5))
    display(df.info())
    display(df.isna().sum().sort_values(ascending=False).head(20))

def safe_parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    possible_date_cols = [col for col in df.columns if 'date' in col.lower() or 'launch' in col.lower() or 'net' in col.lower()]
    for col in possible_date_cols:
        try:
            df[col + '_parsed'] = pd.to_datetime(df[col], errors='coerce', utc=True)
        except Exception:
            continue
    return df

In [ ]:
df = load_space_data(DATA_PATH)
df = safe_parse_dates(df)
describe_dataset(df)

In [ ]:
def clean_space_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

    if 'net_parsed' in df.columns:
        df['launch_date'] = df['net_parsed']
    elif 'launch_date_parsed' in df.columns:
        df['launch_date'] = df['launch_date_parsed']
    elif 'window_start_parsed' in df.columns:
        df['launch_date'] = df['window_start_parsed']
    elif 'date' in df.columns:
        df['launch_date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)

    if 'cost' in df.columns:
        df['cost_usd'] = pd.to_numeric(df['cost'], errors='coerce')
    elif 'mission_cost' in df.columns:
        df['cost_usd'] = pd.to_numeric(df['mission_cost'], errors='coerce')

    if 'status' in df.columns:
        df['status_clean'] = df['status'].astype(str).str.lower().str.strip()
        df['success'] = df['status_clean'].str.contains('success|ok|complete', na=False)
        df['failure'] = df['status_clean'].str.contains('fail|failure|partial|partial_failure', na=False)
    elif 'outcome' in df.columns:
        df['status_clean'] = df['outcome'].astype(str).str.lower().str.strip()
        df['success'] = df['status_clean'].str.contains('success|completed', na=False)
        df['failure'] = df['status_clean'].str.contains('failure|failed', na=False)
    else:
        df['status_clean'] = np.nan
        df['success'] = np.nan
        df['failure'] = np.nan

    if 'country' not in df.columns and 'launch_service_provider_country' in df.columns:
        df['country'] = df['launch_service_provider_country']
    elif 'country_code' in df.columns and 'country' not in df.columns:
        df['country'] = df['country_code']

    if 'launch_service_provider_name' in df.columns:
        df['organization'] = df['launch_service_provider_name']
    elif 'rocket' in df.columns and 'launch_service_provider' in df.columns:
        df['organization'] = df['launch_service_provider']
    elif 'mission_operator' in df.columns:
        df['organization'] = df['mission_operator']

    if 'launch_date' in df.columns:
        df['launch_year'] = df['launch_date'].dt.year
        df['launch_month'] = df['launch_date'].dt.month_name()
        df['launch_month_number'] = df['launch_date'].dt.month
    else:
        df['launch_year'] = pd.NA
        df['launch_month'] = pd.NA

    df['country'] = df.get('country', pd.NA).astype(str).replace('nan', pd.NA)
    df['organization'] = df.get('organization', pd.NA).astype(str).replace('nan', pd.NA)

    return df

cleaned = clean_space_data(df)
display(cleaned[['launch_date', 'launch_year', 'launch_month', 'country', 'organization', 'cost_usd', 'status_clean']].head(8))

In [ ]:
def plot_launches_per_year(df: pd.DataFrame):
    yearly = df.dropna(subset=['launch_year']).groupby('launch_year', as_index=False).size()
    fig = px.line(yearly, x='launch_year', y='size', markers=True, title='Launches per year')
    fig.update_layout(xaxis_title='Year', yaxis_title='Number of launches')
    fig.show()

def plot_top_countries_by_year(df: pd.DataFrame, year: int | None = None):
    sub = df.dropna(subset=['launch_year', 'country'])
    if year is not None:
        sub = sub[sub['launch_year'] == year]
    top = sub['country'].value_counts().rename_axis('country').reset_index(name='launches').head(10)
    title_text = f"Top 10 countries by number of launches in {year}" if year else 'Top 10 countries by number of launches'
    fig = px.bar(top, x='country', y='launches', title=title_text)
    fig.update_layout(xaxis_title='Country', yaxis_title='Launches')
    fig.show()

def plot_cost_trend(df: pd.DataFrame):
    if 'cost_usd' not in df.columns or df['cost_usd'].isna().all():
        print('No valid cost column to plot.')
        return
    cost = df.dropna(subset=['launch_year', 'cost_usd']).groupby('launch_year', as_index=False)['cost_usd'].median()
    fig = px.line(cost, x='launch_year', y='cost_usd', markers=True, title='Median mission cost per year')
    fig.update_layout(xaxis_title='Year', yaxis_title='Cost (USD)')
    fig.show()

def plot_monthly_launches(df: pd.DataFrame):
    if 'launch_month_number' not in df.columns:
        print('No launch date available to analyze by month.')
        return
    month_counts = df.groupby(['launch_month_number', 'launch_month'], as_index=False).size()
    month_counts = month_counts.sort_values('launch_month_number')
    fig = px.bar(month_counts, x='launch_month', y='size', title='Launches by month of year')
    fig.update_layout(xaxis_title='Month', yaxis_title='Launches')
    fig.show()

def plot_success_rate(df: pd.DataFrame):
    if 'success' not in df.columns:
        print('No success/failure data available.')
        return
    status = df.dropna(subset=['launch_year', 'success']).groupby('launch_year', as_index=False)['success'].mean()
    status['success_rate'] = status['success'] * 100
    fig = px.line(status, x='launch_year', y='success_rate', markers=True, title='Average success rate per year')
    fig.update_layout(xaxis_title='Year', yaxis_title='Success rate (%)')
    fig.show()

plot_launches_per_year(cleaned)
plot_top_countries_by_year(cleaned)
plot_monthly_launches(cleaned)
plot_cost_trend(cleaned)
plot_success_rate(cleaned)

## Next steps

- Adjust `DATA_PATH` to the real CSV filename if your file is not named `space_missions.csv`.
- Review the columns table and add specific cleaning steps for fields like `vehicle`, `mission_type`, `pad`, `agency`, `cost`, `status`, and `country`.
- Create a sunburst chart for `country` > `organization` > `launch_year` if the dataset contains those columns.
- Produce a table of the top 10 organizations by mission count and another table for the top 10 mission costs.
- If the dataset includes coordinates, add a heatmap or choropleth of launches by country.